# SOMA — Ablation: Necessity Signals

**Wakasa Labs · Nairobi, Kenya · March 2026**

Removes each necessity signal (N1, N2, N3) one at a time to validate
that each contributes to the overall BT performance.

This produces **Table 2** of Paper 1.

Expected results:
- Full SOMA: BT ≈ -0.038, K ≈ 7
- No N1: BT ≈ -0.065, K ≈ 9 (over-spawns on lr-decay plateaus)
- No N2: BT ≈ -0.071, K ≈ 10 (doesn't detect subspace gaps)
- No N3: BT ≈ -0.058, K ≈ 8 (triggers on noise)
- No RL: BT ≈ -0.047, K ≈ 8 (close but less efficient)

In [4]:
# Clone SOMA repo if not exists and add to path
import os
import sys
import importlib

if not os.path.exists('soma_research'):
    !git clone https://github.com/LensenWakasa/SOMA-research.git soma_research

# Ensure we have the absolute path
repo_path = os.path.abspath('soma_research')

# Add to sys.path and move to front
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# Force a path refresh for the current process
importlib.invalidate_caches()

# Verify import with full path checking
try:
    import soma
    print(f'Soma package found at: {soma.__file__}')
    from soma.core.necessity import SomaNecessity
    from soma.core.grow import SomaGrow
    from soma.core.learn import SomaLearn
    print('SOMA modules loaded successfully')
except ImportError as e:
    print(f'Import failed: {e}')
    print('Current sys.path:', sys.path)

Cloning into 'soma_research'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 147 (delta 61), reused 109 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 203.63 KiB | 1.62 MiB/s, done.
Resolving deltas: 100% (61/61), done.
Soma package found at: /content/soma_research/soma/__init__.py
SOMA modules loaded successfully


In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


In [5]:
import sys
import os
import argparse
import pandas as pd

# Add parent directory to path to import soma
sys.path.insert(0, os.path.abspath('..'))

from soma.experiments.run_permuted_mnist import run_experiment

base_args = {
    "n_tasks": 10,
    "n_train": 1000,
    "n_test": 200,
    "device": device,
    "seed": 42,
    "no_rl": False,
    "disable_n1": False,
    "disable_n2": False,
    "disable_n3": False,
}

variants = [
    ("Full SOMA", {}),
    ("No N1", {"disable_n1": True}),
    ("No N2", {"disable_n2": True}),
    ("No N3", {"disable_n3": True}),
    ("No RL", {"no_rl": True}),
]

results = []

print("Running ablation experiments...")
for name, kwargs in variants:
    print(f"\n--- {name} ---")
    current_args = base_args.copy()
    current_args.update(kwargs)
    args = argparse.Namespace(**current_args)

    result = run_experiment(args)

    results.append({
        "Variant": name,
        "BT": result['backward_transfer'],
        "K": result['final_k']
    })

print("\n--- Ablation Results (Table 2) ---")
df = pd.DataFrame(results)
print(df.to_string(index=False))

Running ablation experiments...

--- Full SOMA ---
=== SOMA Experiment 1: Permuted MNIST ===
Tasks: 10, Train: 1000, Test: 200
Device: cuda, Seed: 42

Generating permuted MNIST tasks...


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 491kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.60MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.20MB/s]


[Task 1/10]
  Action: SPAWN(cold)  K: 0->1  Acc: 0.815  BT: 0.0000
[Task 2/10]
  Action: SPAWN(cold)  K: 1->2  Acc: 0.835  BT: 0.0000
[Task 3/10]
  Action: SPAWN_NEW  K: 2->3  Acc: 0.875  BT: 0.0000
[Task 4/10]
  Action: SPAWN_NEW  K: 3->4  Acc: 0.895  BT: 0.0000
[Task 5/10]
  Action: SKIP  K: 4->4  Acc: 0.100  BT: 0.0000
[Task 6/10]
  Action: SPAWN_NEW  K: 4->5  Acc: 0.830  BT: 0.0050
[Task 7/10]
  Action: SPAWN_NEW  K: 5->6  Acc: 0.875  BT: 0.0042
[Task 8/10]
  Action: SPAWN_NEW  K: 6->7  Acc: 0.815  BT: 0.0036
[Task 9/10]
  Action: SKIP  K: 7->7  Acc: 0.065  BT: 0.0031
[Task 10/10]
  Action: SKIP  K: 7->7  Acc: 0.065  BT: 0.0028

=== SOMA Summary ===
  backward_transfer        : 0.0028
  forward_transfer         : 0.0000
  final_k                  : 7
  spawn_count              : 7
  merge_count              : 0
  tasks_completed          : 10
  target_bt_met            : True

PASS: BT=0.0028 K=7

--- No N1 ---
=== SOMA Experiment 1: Permuted MNIST ===
Tasks: 10, Train: 1000, Test: